In [20]:
import pandas as pd
import numpy as np
from collections import defaultdict
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from xgboost import XGBClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import f1_score, roc_auc_score
import joblib

In [21]:
diagnoses = pd.read_csv('../data/DIAGNOSES_ICD.csv', usecols=['hadm_id', 'icd9_code'])
admissions = pd.read_csv('../data/ADMISSIONS.csv', usecols=['hadm_id', 'subject_id'])
patients = pd.read_csv('../data/PATIENTS.csv', usecols=['subject_id', 'gender', 'dob'])
labevents = pd.read_csv('../data/LABEVENTS.csv', usecols=['subject_id', 'itemid', 'valuenum'])
chartevents = pd.read_csv('../data/CHARTEVENTS.csv', usecols=['subject_id', 'itemid', 'valuenum'])
d_items = pd.read_csv('../data/D_ITEMS.csv')
d_labitems = pd.read_csv('../data/D_LABITEMS.csv')

Связываем госпитализации с пациентами

In [22]:
hadm_to_subj = admissions.set_index('hadm_id')['subject_id'].to_dict()
diagnoses['subject_id'] = diagnoses['hadm_id'].map(hadm_to_subj)
diagnoses = diagnoses.dropna(subset=['subject_id'])

In [23]:
def is_cardio_icd9(icd9):
    if pd.isna(icd9):
        return False
    icd9_str = str(icd9).strip()
    try:
        code_num = icd9_str[:3]
        if code_num.isdigit():
            code_int = int(code_num)
            return 390 <= code_int <= 459
    except:
        pass
    return False

In [24]:
def map_icd9_to_category(icd9):
    if pd.isna(icd9):
        return None
    icd9_str = str(icd9).strip()
    try:
        code_num = icd9_str[:3]
        if code_num.isdigit():
            code_int = int(code_num)
            if 390 <= code_int <= 392:
                return 'Acute_rheumatic_fever'
            elif 393 <= code_int <= 398:
                return 'Chronic_rheumatic_heart_disease'
            elif 401 <= code_int <= 405:
                return 'Hypertension'
            elif 410 <= code_int <= 414:
                return 'Ischemic_heart_disease'
            elif 420 <= code_int <= 429:
                return 'Other_heart_disease'
            elif 430 <= code_int <= 438:
                return 'Cerebrovascular_disease'
            elif 440 <= code_int <= 449:
                return 'Arterial_disease'
            elif 451 <= code_int <= 459:
                return 'Venous_disease'
            else:
                return 'Other_cardio'
    except:
        pass
    return 'Other_cardio'


In [25]:
cardio_diagnoses = diagnoses[diagnoses['icd9_code'].apply(is_cardio_icd9)].copy()
cardio_diagnoses['category'] = cardio_diagnoses['icd9_code'].apply(map_icd9_to_category)

patient_categories = cardio_diagnoses.groupby('subject_id')['category'].apply(list).reset_index()
print(f"Уникальных пациентов с сердечно-сосудистыми диагнозами: {len(patient_categories)}")

Уникальных пациентов с сердечно-сосудистыми диагнозами: 89


In [26]:
def get_itemids_for_features(df_items, df_labitems, feature_groups):
    feature_itemids = {}
    for feature, keywords in feature_groups.items():
        itemids = []
        mask = df_items['label'].str.lower().str.contains('|'.join(keywords), na=False)
        itemids.extend(df_items.loc[mask, 'itemid'].tolist())
        mask = df_labitems['label'].str.lower().str.contains('|'.join(keywords), na=False)
        itemids.extend(df_labitems.loc[mask, 'itemid'].tolist())
        feature_itemids[feature] = list(set(itemids))
    return feature_itemids

feature_groups = {
    'heart_rate': ['heart rate', 'pulse'],
    'sys_bp': ['systolic', 'arterial bp systolic'],
    'dia_bp': ['diastolic', 'arterial bp diastolic'],
    'mean_bp': ['mean arterial pressure', 'arterial bp mean'],
    'resp_rate': ['respiratory rate', 'respiration'],
    'temperature': ['temperature'],
    'spo2': ['oxygen saturation', 'spo2'],
    'glucose': ['glucose'],
    'potassium': ['potassium'],
    'sodium': ['sodium'],
    'creatinine': ['creatinine'],
    'hematocrit': ['hematocrit'],
    'hemoglobin': ['hemoglobin'],
    'wbc': ['white blood cell', 'wbc'],
    'platelets': ['platelet'],
    'chloride': ['chloride'],
    'bicarbonate': ['bicarbonate', 'co2'],
    'bun': ['blood urea nitrogen', 'bun'],
    'inr': ['inr', 'prothrombin time'],
    'alt': ['alt', 'alanine aminotransferase'],
    'ast': ['ast', 'aspartate aminotransferase'],
    'bilirubin': ['bilirubin'],
    'ldh': ['lactate dehydrogenase', 'ldh'],
    'ck': ['creatine kinase', 'ck'],
    'ckmb': ['ckmb', 'creatine kinase mb'],
    'troponin': ['troponin'],
    'nt_probnp': ['nt-probnp', 'bnp', 'probnp'],
}

feature_itemids = get_itemids_for_features(d_items, d_labitems, feature_groups)

In [27]:
def aggregate_patient_measurements(df, patient_ids, feature_itemids):
    result = defaultdict(lambda: defaultdict(list))
    for _, row in df.iterrows():
        subj = row['subject_id']
        if subj not in patient_ids:
            continue
        itemid = row['itemid']
        value = row['valuenum']
        if pd.isna(value):
            continue
        for feature, ids in feature_itemids.items():
            if itemid in ids:
                result[subj][feature].append(value)
                break
    return result

In [28]:
patient_ids = set(patient_categories['subject_id'])
print(f"Обрабатываем {len(patient_ids)} пациентов")

lab_data = aggregate_patient_measurements(labevents, patient_ids, feature_itemids)
print(f"Лабораторные данные для {len(lab_data)} пациентов")

chart_data = aggregate_patient_measurements(chartevents, patient_ids, feature_itemids)
print(f"Витальные данные для {len(chart_data)} пациентов")


Обрабатываем 89 пациентов
Лабораторные данные для 89 пациентов
Витальные данные для 88 пациентов


In [29]:
X_rows = []
for subj in patient_ids:
    row = {'subject_id': subj}
    combined = defaultdict(list)
    if subj in lab_data:
        for feature, values in lab_data[subj].items():
            combined[feature].extend(values)
    if subj in chart_data:
        for feature, values in chart_data[subj].items():
            combined[feature].extend(values)
    for feature in feature_groups.keys():
        values = combined.get(feature, [])
        if values:
            row[f'{feature}_mean'] = np.mean(values)
            row[f'{feature}_min'] = np.min(values)
            row[f'{feature}_max'] = np.max(values)
            row[f'{feature}_median'] = np.median(values)
            row[f'{feature}_std'] = np.std(values) if len(values) > 1 else 0
            row[f'{feature}_count'] = len(values)
        else:
            row[f'{feature}_mean'] = np.nan
            row[f'{feature}_min'] = np.nan
            row[f'{feature}_max'] = np.nan
            row[f'{feature}_median'] = np.nan
            row[f'{feature}_std'] = np.nan
            row[f'{feature}_count'] = 0
    X_rows.append(row)

X_df = pd.DataFrame(X_rows)
X_df = X_df.set_index('subject_id')
print(f"Матрица признаков: {X_df.shape}")

Матрица признаков: (89, 162)


In [34]:
patients_info = patients.set_index('subject_id')
for col in ['gender', 'dob']:
    if col in X_df.columns:
        X_df = X_df.drop(columns=[col])
X_df = X_df.join(patients_info[['gender', 'dob']], how='left')
X_df['dob'] = pd.to_datetime(X_df['dob'], errors='coerce')

admissions = pd.read_csv('../data/ADMISSIONS.csv', usecols=['hadm_id', 'subject_id'])
admissions_sample = pd.read_csv('../data/ADMISSIONS.csv', nrows=5)
time_candidates = [col for col in admissions_sample.columns if 'admit' in col.lower() or 'time' in col.lower()]
if not time_candidates:
    print("Доступные столбцы:", admissions_sample.columns.tolist())
    raise ValueError("Не найден столбец с датой поступления")
time_col = time_candidates[0]
print(f"Используем столбец {time_col} для даты поступления")

admissions = pd.read_csv('../data/ADMISSIONS.csv', usecols=['hadm_id', 'subject_id', time_col])
admissions_agg = admissions.groupby('subject_id')[time_col].min().reset_index()
admissions_agg = admissions_agg.rename(columns={time_col: 'admittime'})
admissions_agg['admittime'] = pd.to_datetime(admissions_agg['admittime'])
admissions_agg = admissions_agg.set_index('subject_id')

X_df = X_df.join(admissions_agg, how='left')
X_df['age'] = X_df['admittime'].dt.year - X_df['dob'].dt.year
X_df['gender'] = (X_df['gender'] == 'M').astype(int)
X_df = X_df.drop(columns=['dob', 'admittime'])

Используем столбец admittime для даты поступления


In [35]:
patient_cat_dict = patient_categories.set_index('subject_id')['category'].to_dict()
common_patients = set(X_df.index) & set(patient_cat_dict.keys())
X_filtered = X_df.loc[list(common_patients)]
y_labels = [patient_cat_dict[pid] for pid in common_patients]

mlb = MultiLabelBinarizer()
y_full = mlb.fit_transform(y_labels)
class_names = mlb.classes_

print(f"Всего уникальных категорий: {len(class_names)}")
print("Категории:")
for i, cat in enumerate(class_names):
    print(f"  {i+1}. {cat}")
print(f"Форма X: {X_filtered.shape}, форма y: {y_full.shape}")

Всего уникальных категорий: 8
Категории:
  1. Arterial_disease
  2. Cerebrovascular_disease
  3. Chronic_rheumatic_heart_disease
  4. Hypertension
  5. Ischemic_heart_disease
  6. Other_cardio
  7. Other_heart_disease
  8. Venous_disease
Форма X: (89, 164), форма y: (89, 8)


In [36]:
X_train, X_test, y_train, y_test = train_test_split(
    X_filtered, y_full, test_size=0.2, random_state=42
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

Train: (71, 164), Test: (18, 164)


Обработка пропусков

In [38]:
numeric_features = X_train.select_dtypes(include=[np.number]).columns.tolist()
print(f"Числовых признаков: {len(numeric_features)}")

imputer = SimpleImputer(strategy='median')
X_train_imp = imputer.fit_transform(X_train[numeric_features])
X_test_imp = imputer.transform(X_test[numeric_features])

X_train_imp = pd.DataFrame(X_train_imp, columns=numeric_features, index=X_train.index)
X_test_imp = pd.DataFrame(X_test_imp, columns=numeric_features, index=X_test.index)

Числовых признаков: 164


/opt/homebrew/lib/python3.14/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['ckmb_mean' 'ckmb_min' 'ckmb_max' 'ckmb_median' 'ckmb_std']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/opt/homebrew/lib/python3.14/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['ckmb_mean' 'ckmb_min' 'ckmb_max' 'ckmb_median' 'ckmb_std']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


ValueError: Shape of passed values is (71, 159), indices imply (71, 164)

Обучение модели

In [ ]:
base_model = XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=42,
    eval_metric='logloss',
    use_label_encoder=False
)

multi_model = MultiOutputClassifier(base_model, n_jobs=-1)
multi_model.fit(X_train_imp, y_train)

Оценка модели

In [ ]:
y_pred_proba = np.array([est.predict_proba(X_test_imp)[:, 1] for est in multi_model.estimators_]).T
y_pred = (y_pred_proba > 0.5).astype(int)

f1_micro = f1_score(y_test, y_pred, average='micro')
f1_macro = f1_score(y_test, y_pred, average='macro')
print(f"F1 micro: {f1_micro:.4f}")
print(f"F1 macro: {f1_macro:.4f}")

print("ROC-AUC по категориям:")
aucs = []
for i, name in enumerate(class_names):
    if len(np.unique(y_test[:, i])) > 1:
        auc = roc_auc_score(y_test[:, i], y_pred_proba[:, i])
        aucs.append(auc)
        print(f"  {name}: {auc:.4f}")
    else:
        print(f"  {name}: недостаточно данных")
print(f"Средний ROC-AUC: {np.mean(aucs):.4f}")

In [ ]:
joblib.dump(multi_model, '../models/multi_model_patient.pkl')
joblib.dump(imputer, '../models/imputer_patient.pkl')
joblib.dump(mlb, '../models/mlb_patient.pkl')
joblib.dump(class_names, '../models/class_names_patient.pkl')
joblib.dump(numeric_features, '../models/numeric_features_patient.pkl')
joblib.dump(feature_itemids, '../models/feature_itemids_patient.pkl')